In [1]:
import os
import pandas as pd

CSV_PATH = "/content/100_Unique_QA_Dataset.csv"

if not os.path.exists(CSV_PATH):
    # Fallback for environments outside Google Colab (no bundled dataset)
    sample_qa = [
        ("What is the capital of France", "Paris"),
        ("What is the capital of Japan", "Tokyo"),
        ("What is the largest planet in the solar system", "Jupiter"),
        ("Who wrote Romeo and Juliet", "Shakespeare"),
        ("What is the chemical symbol for gold", "Gold"),
        ("How many continents are there", "Seven"),
        ("What is the tallest mountain in the world", "Everest"),
        ("What language is spoken in Brazil", "Portuguese"),
        ("What is the national currency of Japan", "Yen"),
        ("What is the freezing point of water", "Zero"),
    ]
    df = pd.DataFrame(sample_qa, columns=["question", "answer"])
else:
    df = pd.read_csv(CSV_PATH)

df.head()

,question,answer
0,What is the capital of France,Paris
1,What is the capital of Japan,Tokyo
2,What is the largest planet in the solar system,Jupiter
3,Who wrote Romeo and Juliet,Shakespeare
4,What is the chemical symbol for gold,Gold


In [2]:
def tokenize(text):
  text=text.lower()
  text=text.replace("?","")
  text=text.replace("'","")
  return text.split()

In [3]:
tokenize("What is the capital of France?")

['what', 'is', 'the', 'capital', 'of', 'france']

In [4]:
vocab={
    '<UNK>':0,
}

In [5]:
def build_vocab(row):
  tokenized_ques=tokenize(row["question"])
  tokenized_ans=tokenize(row['answer'])

  merged_token=tokenized_ques+tokenized_ans

  for token in merged_token:
    if token not in vocab:
      vocab[token]=len(vocab)

  print(tokenized_ques,tokenized_ans)

In [6]:
df.apply(build_vocab,axis=1)

['what', 'is', 'the', 'capital', 'of', 'france'] ['paris']
['what', 'is', 'the', 'capital', 'of', 'japan'] ['tokyo']
['what', 'is', 'the', 'largest', 'planet', 'in', 'the', 'solar', 'system'] ['jupiter']
['who', 'wrote', 'romeo', 'and', 'juliet'] ['shakespeare']
['what', 'is', 'the', 'chemical', 'symbol', 'for', 'gold'] ['gold']
['how', 'many', 'continents', 'are', 'there'] ['seven']
['what', 'is', 'the', 'tallest', 'mountain', 'in', 'the', 'world'] ['everest']
['what', 'language', 'is', 'spoken', 'in', 'brazil'] ['portuguese']
['what', 'is', 'the', 'national', 'currency', 'of', 'japan'] ['yen']
['what', 'is', 'the', 'freezing', 'point', 'of', 'water'] ['zero']


0    None
1    None
2    None
3    None
4    None
5    None
6    None
7    None
8    None
9    None
dtype: object

In [7]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'japan': 8,
 'tokyo': 9,
 'largest': 10,
 'planet': 11,
 'in': 12,
 'solar': 13,
 'system': 14,
 'jupiter': 15,
 'who': 16,
 'wrote': 17,
 'romeo': 18,
 'and': 19,
 'juliet': 20,
 'shakespeare': 21,
 'chemical': 22,
 'symbol': 23,
 'for': 24,
 'gold': 25,
 'how': 26,
 'many': 27,
 'continents': 28,
 'are': 29,
 'there': 30,
 'seven': 31,
 'tallest': 32,
 'mountain': 33,
 'world': 34,
 'everest': 35,
 'language': 36,
 'spoken': 37,
 'brazil': 38,
 'portuguese': 39,
 'national': 40,
 'currency': 41,
 'yen': 42,
 'freezing': 43,
 'point': 44,
 'water': 45,
 'zero': 46}

In [8]:
def text_to_indices(text,vocab):
  indexed_text=[]
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text

In [9]:
text_to_indices("What is apple",vocab)

[1, 2, 0]

In [10]:
import torch
from torch.utils.data import Dataset,DataLoader

In [11]:
class QADatatset(Dataset):
  def __init__(self,df,vocab):
    self.df=df
    self.vocab=vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numerical_question=text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numerical_answer=text_to_indices(self.df.iloc[index]['answer'],self.vocab)

    return torch.tensor(numerical_question),torch.tensor(numerical_answer)

In [12]:
dataset=QADatatset(df,vocab)

In [13]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)

In [14]:
for question,answer in dataloader:
  print(question,answer[0])

tensor([[ 1,  2,  3, 32, 33, 12,  3, 34]]) tensor([35])
tensor([[1, 2, 3, 4, 5, 6]]) tensor([7])
tensor([[16, 17, 18, 19, 20]]) tensor([21])
tensor([[ 1, 36,  2, 37, 12, 38]]) tensor([39])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([9])
tensor([[ 1,  2,  3, 22, 23, 24, 25]]) tensor([25])
tensor([[ 1,  2,  3, 10, 11, 12,  3, 13, 14]]) tensor([15])
tensor([[26, 27, 28, 29, 30]]) tensor([31])
tensor([[ 1,  2,  3, 43, 44,  5, 45]]) tensor([46])
tensor([[ 1,  2,  3, 40, 41,  5,  8]]) tensor([42])


In [15]:
import torch.nn as nn


In [16]:
class SimpleRNN(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn=nn.RNN(50,64,batch_first=True)
    self.fc=nn.Linear(64,vocab_size)

  def forward(self,x):
    embedded_question=self.embedding(x)
    hidden,final=self.rnn(embedded_question)
    output=self.fc(final.squeeze(0))
    return output

In [17]:
learning_rate=0.001

In [18]:
epochs=20


In [19]:
model=SimpleRNN(len(vocab))

In [20]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)


In [21]:
for epoch in range(epochs):
  total_loss=0
  for question, answer in dataloader:
    optimizer.zero_grad()
    output=model(question)
    loss=criterion(output,answer[0])
    loss.backward()
    optimizer.step()
    total_loss=total_loss+loss.item()
  print(f"Epoch {epoch+1},Loss:{total_loss:4f}")

Epoch 1,Loss:40.014256
Epoch 2,Loss:34.995333
Epoch 3,Loss:30.968875
Epoch 4,Loss:26.995471
Epoch 5,Loss:23.281079
Epoch 6,Loss:19.895576
Epoch 7,Loss:16.810252
Epoch 8,Loss:14.086694
Epoch 9,Loss:11.815003
Epoch 10,Loss:9.753784
Epoch 11,Loss:8.152362
Epoch 12,Loss:6.794323
Epoch 13,Loss:5.709427
Epoch 14,Loss:4.821039
Epoch 15,Loss:4.157919
Epoch 16,Loss:3.591576
Epoch 17,Loss:3.124130
Epoch 18,Loss:2.748155
Epoch 19,Loss:2.435332
Epoch 20,Loss:2.172453


In [22]:
def predict(model,question,threshold=0.5):
  numerical_ques=text_to_indices(question,vocab)
  question_tensor=torch.tensor(numerical_ques).unsqueeze(0)
  output=model(question_tensor)
  prob=torch.nn.functional.softmax(output,dim=1)
  value,index=torch.max(prob,dim=1)
  if value<threshold:
    print("IDK")
  print(list(vocab.keys())[index])

In [23]:
predict(model,"WHat is capital of France")

paris
